### Dataset and Task Metadata

In [25]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="regensburg_pediatric_appendicitis",
    dataset_year="2021",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://doi.org/10.5281/zenodo.7711412",
    download_description="""
We get the newest version of the tabular data from Zenodo.


wget https://zenodo.org/records/7711412/files/app_data.xlsx?download=1
mkdir -p local-data-warehouse/regensburg_pediatric_appendicitis && mv app_data.xlsx?download=1 local-data-warehouse/regensburg_pediatric_appendicitis/
""",
    # References
    academic_reference_bibtex=r"""@article{marcinkevivcs2024interpretable,
  title={Interpretable and intervenable ultrasonography-based machine learning models for pediatric appendicitis},
  author={Marcinkevi{\v{c}}s, Ri{\v{c}}ards and Wolfertstetter, Patricia Reis and Klimiene, Ugne and Chin-Cheong, Kieran and Paschke, Alyssia and Zerres, Julia and Denzinger, Markus and Niederberger, David and Wellmann, Sven and Ozkan, Ece and others},
  journal={Medical image analysis},
  volume={91},
  pages={103042},
  year={2024},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="marcinkevivcs2024interpretable",
    license="CC-BY-NC-4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the tabular meta-data used in the paper. We create only one task out of the dataset using one of the three available classes. We note that the main use case of the paper was image-based modelling and comparing to tabular baselines with image-to-tabular-radiometric. The authors also published the tabular data, which we use here.

- Note, for this dataset is was hard to choose the label given the tabular data. We picked "Severity" in the end. "Diagnosis" is a noisy ground truth for all cases but the ones that had surgery. It was either deterministically determined by the AS score, or during surgery. "Management" reflects the decision and opinion of senior pediatric surgeon. Only "Severity" represents a ground truth as all cases that were server, were either clearly servery or did not even have appendicitis.
 - Warning: We tried to clean the data as good as possible but we suspect it might still contain some features leaking the severity, but we lack the medical expertise to identify them. Thus, I would not be surprised if we had to remove this dataset later again.
- The data contains string data, but only for a very small number of patients and the text is often seemingly very useless. The features add most likely no value, but we keep them in for the pipeline/model to ignore if needed.
- We drop the few patients that did not have an ultrasound to avoid them being leaking due to missing features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Severity",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Severity",
)

## Preprocessing

In [26]:
import pandas as pd
import numpy as np

df = pd.read_excel(dataset_mold.path / "app_data.xlsx?download=1", sheet_name=0)
print("Loaded data shape:", df.shape)

# Drop samples w/o utrasounds
df = df[df["US_Performed"] == "yes"]

as_cat_type = [
    "Sex",
    "Severity",
    "Peritonitis",
    "Migratory_Pain",
    "Lower_Right_Abd_Pain",
    "Contralateral_Rebound_Tenderness",
    "Ipsilateral_Rebound_Tenderness",
    "Coughing_Pain",
    "Psoas_Sign",
    "Nausea",
    "Loss_of_Appetite",
    "Dysuria",
    "Stool",
    "Ketones_in_Urine",
    "RBC_in_Urine",
    "WBC_in_Urine",
    "Neutrophilia",
    "Appendix_on_US",
    "Free_Fluids",
    "Appendix_Wall_Layers",
    "Target_Sign",
    "Perfusion",
    "Surrounding_Tissue_Reaction",
    "Pathological_Lymph_Nodes",
    "Bowel_Wall_Thickening",
    "Ileus",
    "Coprostasis",
    "Meteorism",
    "Enteritis",
    "Appendicolith",
    "Perforation",
    "Appendicular_Abscess",
    "Conglomerate_of_Bowel_Loops",
]
as_string_type = [
    "Lymph_Nodes_Location",
    "Abscess_Location",
    "Gynecological_Findings"
]
df = df.drop(columns=[
    # Other
    "US_Performed", # constant after dropping other cases
    "US_Number", # rel us_performed
    "Length_of_Stay", # only available at discharge (so after diagnosis)
    # Other target labels Diagnosis / Management / Severity
    "Diagnosis_Presumptive",
    "Diagnosis",
    "Management",
])

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (782, 58)


## Data Checks

In [27]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 763
Columns: 52
Use sampling: False (sample size: 763)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Age', 'BMI', 'Neutrophil_Percentage', 'Weight', 'Thrombocyte_Count', 'WBC_Count', 'Height', 'RBC_Count', 'CRP', 'Appendix_Diameter']
Rows remaining as candidates after top-10 filter: 0 (of 763)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [28]:
# Sample Rows
df_head

,Age,BMI,Sex,Height,Weight,Severity,Alvarado_Score,Paedriatic_Appendicitis_Score,Appendix_on_US,Appendix_Diameter,Migratory_Pain,Lower_Right_Abd_Pain,Contralateral_Rebound_Tenderness,Coughing_Pain,Nausea,Loss_of_Appetite,Body_Temperature,WBC_Count,Neutrophil_Percentage,Segmented_Neutrophils,Neutrophilia,RBC_Count,Hemoglobin,RDW,Thrombocyte_Count,Ketones_in_Urine,RBC_in_Urine,WBC_in_Urine,CRP,Dysuria,Stool,Peritonitis,Psoas_Sign,Ipsilateral_Rebound_Tenderness,Free_Fluids,Appendix_Wall_Layers,Target_Sign,Appendicolith,Perfusion,Perforation,Surrounding_Tissue_Reaction,Appendicular_Abscess,Abscess_Location,Pathological_Lymph_Nodes,Lymph_Nodes_Location,Bowel_Wall_Thickening,Conglomerate_of_Bowel_Loops,Ileus,Coprostasis,Meteorism,Enteritis,Gynecological_Findings
0,12.870637,15.147929,female,162.5,40.0,uncomplicated,7.0,4.0,no,NaN,no,yes,yes,no,no,no,38.6,13.6,84.0,NaN,yes,5.16,15.5,12.3,259.0,NaN,NaN,NaN,34.0,no,diarrhea,no,no,no,no,NaN,NaN,NaN,NaN,NaN,NaN,no,<NA>,yes,mesenterial,no,NaN,no,NaN,NaN,NaN,<NA>
1,14.425736,20.530000,female,150.5,46.5,uncomplicated,2.0,4.0,yes,5.0,no,yes,no,yes,no,no,37.0,7.7,63.2,NaN,no,4.77,13.0,11.9,230.0,no,no,no,0.0,no,normal,no,yes,no,yes,NaN,no,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,yes,NaN,NaN,Ovarialzyste
2,11.548255,17.529432,female,152.0,40.5,uncomplicated,9.0,7.0,yes,10.0,no,yes,yes,yes,yes,no,37.8,12.6,85.1,NaN,yes,5.04,14.7,12.3,255.0,+++,+,no,5.0,no,diarrhea,local,no,no,yes,NaN,NaN,NaN,NaN,no,yes,no,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
3,9.940000,15.300000,male,140.0,29.5,uncomplicated,8.0,6.0,no,NaN,yes,yes,yes,no,no,no,37.4,18.6,75.2,NaN,yes,5.29,14.3,13.5,433.0,+++,+,no,37.0,no,constipation,local,no,no,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,9.193703,15.432099,female,144.0,32.0,uncomplicated,10.0,9.0,yes,7.5,yes,yes,yes,yes,yes,yes,37.8,13.6,79.2,NaN,yes,4.93,12.8,14.1,302.0,no,+,no,0.0,no,constipation,no,yes,yes,no,intact,yes,NaN,NaN,no,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [29]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Conglomerate_of_Bowel_Loops,category,720.0,94.36,2.0,"no, yes"
1,Ileus,category,703.0,92.14,2.0,"no, yes"
2,Perfusion,category,701.0,91.87,4.0,"hyperperfused, hypoperfused, no, present"
3,Enteritis,category,697.0,91.35,2.0,"yes, no"
4,Appendicolith,category,694.0,90.96,3.0,"no, yes, suspected"
5,Coprostasis,category,692.0,90.69,2.0,"yes, no"
6,Perforation,category,682.0,89.38,4.0,"no, yes, not excluded, suspected"
7,Appendicular_Abscess,category,679.0,88.99,3.0,"no, yes, suspected"
8,Bowel_Wall_Thickening,category,664.0,87.02,2.0,"yes, no"
9,Target_Sign,category,626.0,82.04,2.0,"yes, no"


In [30]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,763.0,11.313140,3.544573,0.000000,18.360000
BMI,738.0,18.866877,4.382206,7.827983,38.156221
Height,739.0,147.864953,19.804442,53.000000,192.000000
Weight,761.0,43.003824,17.393327,3.960000,103.000000
Alvarado_Score,720.0,5.922222,2.153254,0.000000,10.000000
Paedriatic_Appendicitis_Score,720.0,5.259722,1.957984,0.000000,10.000000
Appendix_Diameter,496.0,7.755645,2.539182,2.700000,17.000000
Body_Temperature,761.0,37.405913,0.907442,26.900000,40.200000
WBC_Count,759.0,12.680303,5.354096,2.600000,37.700000
Neutrophil_Percentage,667.0,71.841679,14.447242,27.200000,97.700000


In [31]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                           rank                                      
Abscess_Location                 1                       <NA>    750   98.3
                                 2                    Douglas      6   0.79
                                 3         rechter Unterbauch      2   0.26
                                 4            perityphlitisch      1   0.13
                                 5     an den M. psoas rechts      1   0.13
Appendicolith                    1                       <NA>    694  90.96
                                 2                         no     33   4.33
                                 3                        yes     33   4.33
                                 4                  suspected      3   0.39
Appendicular_Abscess             1                       <NA>    679  88.99
                                 2                         no     65   8.52
                                 3                        yes     18   2.36
                                 4                  suspected      1   0.13
Appendix_Wall_Layers             1                       <NA>    546  71.56
                                 2                     intact    132   17.3
                                 3                     raised     75   9.83
                                 4           partially raised      9   1.18
                                 5                      upset      1   0.13
Appendix_on_US                   1                        yes    500  65.53
                                 2                         no    262  34.34
                                 3                       <NA>      1   0.13
Bowel_Wall_Thickening            1                       <NA>    664  87.02
                                 2                        yes     55   7.21
                                 3                         no     44   5.77
Conglomerate_of_Bowel_Loops      1                       <NA>    720  94.36
                                 2                         no     22   2.88
                                 3                        yes     21   2.75
Contralateral_Rebound_Tenderness 1                         no    462  60.55
                                 2                        yes    293   38.4
                                 3                       <NA>      8   1.05
Coprostasis                      1                       <NA>    692  90.69
                                 2                        yes     46   6.03
                                 3                         no     25   3.28
Coughing_Pain                    1                         no    538  70.51
                                 2                        yes    217  28.44
                                 3                       <NA>      8   1.05
Dysuria                          1                         no    697  91.35
                                 2                        yes     44   5.77
                                 3                       <NA>     22   2.88
Enteritis                        1                       <NA>    697  91.35
                                 2                        yes     51   6.68
                                 3                         no     15   1.97
Free_Fluids                      1                         no    409   53.6
                                 2                        yes    309   40.5
                                 3                       <NA>     45    5.9
Gynecological_Findings           1                       <NA>    737  96.59
                                 2                      keine      9   1.18
                                 3               Ovarialzyste      4   0.52
                                 4              Ovarialzysten      2   0.26
                                 5       V. a. Ovarialtorsion      1   0.13
Ileus                            1                       <NA>    703  92.14
                            

In [32]:
# Target Distribution
target_df

,count,pct
Severity,,
uncomplicated,648,84.93
complicated,115,15.07


## Task Curation

In [33]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [34]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [35]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c717a-885a-7ecd-a500-85c3cd09fd6f
6996fe03ff66d0587002094470a9ea6800e79eef1d83ba5bc6498441a0e1e74f
